# HealthBot - Sistema de Aprendizaje Interactivo con LangGraph

Este notebook implementa un sistema educativo que:
1. Recibe un tema de aprendizaje
2. Busca información con Tavily
3. Genera un resumen en español (basado únicamente en Tavily)
4. Crea una pregunta tipo quiz
5. Califica la respuesta del usuario
6. Permite reiniciar o salir

**Stack tecnológico:**
- LangGraph (orquestación del flujo)
- OpenAI (generación de texto)
- Tavily (búsqueda web)
- Langchain (integración)


In [ ]:
# Instalación de dependencias
import subprocess
import sys

packages = [
    "langgraph",
    "langchain",
    "langchain-google-genai",
    "langchain-community",
    "python-dotenv"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [ ]:
# Importaciones
import os
from typing import TypedDict, Any
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langchain_core.messages import HumanMessage, AIMessage

# Cargar variables de entorno
from dotenv import load_dotenv
import sys

# Especificar ruta absoluta del .env
env_path = r"c:\Users\pablo\Desktop\HealthBot-demo\.env"
load_dotenv(dotenv_path=env_path)

# Configuración de APIs
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("="*60)
print("🔧 CONFIGURACIÓN DE HEALTHBOT")
print("="*60)

if OPENAI_API_KEY:
    print("✅ OPENAI_API_KEY configurada (GPT-4)")
else:
    print("⚠️ ADVERTENCIA: OPENAI_API_KEY no configurada")
    print("   Obtén tu clave en: https://platform.openai.com/api-keys")

if TAVILY_API_KEY:
    print("✅ TAVILY_API_KEY configurada")
else:
    print("⚠️ ADVERTENCIA: TAVILY_API_KEY no configurada")
    print("   Visite https://tavily.com para obtener una clave")
print("="*60)

# Inicializar herramientas solo si tenemos las keys
if OPENAI_API_KEY and TAVILY_API_KEY:
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        api_key=OPENAI_API_KEY,
        temperature=0.7
    )
    search_tool = TavilySearchResults(max_results=5, api_key=TAVILY_API_KEY)
    print("\n✅ LLM (OpenAI GPT-4) y Tavily inicializados")
    print("   Modelo: gpt-4o-mini (Recomendado por costo/rendimiento)")
else:
    llm = None
    search_tool = None
    print("\n⚠️ Las herramientas serán inicializadas cuando proporciones las keys")

In [ ]:
# Diagnóstico: verificar contenido del .env
import subprocess
result = subprocess.run(["type", r"c:\Users\pablo\Desktop\HealthBot-demo\.env"], shell=True, capture_output=True, text=True)
print("Contenido del .env:")
print(result.stdout)
print("\nVariables detectadas por Python:")
print(f"GOOGLE_API_KEY: {repr(os.getenv('GOOGLE_API_KEY'))}")
print(f"TAVILY_API_KEY: {repr(os.getenv('TAVILY_API_KEY'))}")
print(f"OPENAI_API_KEY: {repr(os.getenv('OPENAI_API_KEY'))}")

In [ ]:
# Definición del Estado compartido
class HealthBotState(TypedDict):
    """Estado compartido del flujo de aprendizaje"""
    topic: str  # Tema ingresado por el usuario
    search_results: dict  # Resultados de Tavily
    summary: str  # Resumen en español (3-4 párrafos)
    quiz_question: str  # Pregunta generada
    user_answer: str  # Respuesta del usuario
    grade: str  # Calificación (A, B, C, D, F)
    justification: str  # Justificación de la calificación
    continue_learning: bool  # ¿Continuar con otro tema?
    current_step: str  # Paso actual del flujo

In [ ]:
# NODO 1: Solicitar tema de aprendizaje
def input_topic(state: HealthBotState) -> HealthBotState:
    """
    Solicita al usuario que ingrese el tema de aprendizaje.
    """
    print("\n" + "="*60)
    print("🎓 BIENVENIDO A HEALTHBOT - Sistema de Aprendizaje Interactivo")
    print("="*60)
    
    topic = input("\n📚 ¿Cuál es el tema que deseas aprender? (ej: Inteligencia Artificial, Historia de la Medicina, etc.)\n➜ ")
    
    state["topic"] = topic.strip()
    state["current_step"] = "search"
    
    print(f"\n✓ Tema seleccionado: {state['topic']}")
    
    return state

In [ ]:
# NODO 2: Buscar información con Tavily
def search_tavily(state: HealthBotState) -> HealthBotState:
    """
    Realiza una búsqueda con Tavily sobre el tema.
    Esta es la ÚNICA fuente de información para el resumen.
    """
    print(f"\n🔍 Buscando información sobre: {state['topic']}")
    
    try:
        # Ejecutar búsqueda
        results = search_tool.invoke(state["topic"])
        state["search_results"] = results
        
        print(f"✓ Se encontraron {len(results)} fuentes de información")
        
        # Mostrar resumen de fuentes
        for i, result in enumerate(results, 1):
            source = result.get("source", "Fuente desconocida")
            print(f"  {i}. {source}")
            
    except Exception as e:
        print(f"❌ Error en búsqueda: {str(e)}")
        state["search_results"] = []
    
    state["current_step"] = "generate_summary"
    return state

In [ ]:
# NODO 3: Generar resumen en español
def generate_summary(state: HealthBotState) -> HealthBotState:
    """
    Genera un resumen en español de 3-4 párrafos basado ÚNICAMENTE
    en los resultados de Tavily. NO utiliza conocimiento previo.
    """
    print(f"\n📝 Generando resumen sobre: {state['topic']}")
    
    if not state["search_results"]:
        print("❌ No hay resultados de búsqueda para generar resumen")
        state["summary"] = "No se pudo generar resumen debido a falta de resultados."
        return state
    
    # Construir contexto EXCLUSIVAMENTE desde Tavily
    search_context = "\n".join([f"Fuente: {r.get('source', 'N/A')}\nContenido: {r.get('content', '')}" 
                                 for r in state["search_results"]])
    
    # Prompt que asegura que solo usa búsqueda
    prompt = f"""Eres un asistente educativo que ÚNICAMENTE puedes usar información
de una búsqueda web. NO tienes conocimiento previo sobre el tema.

Resultados de búsqueda sobre '{state['topic']}':\n\n{search_context}\n\nTarea: Genera un resumen en español de exactamente 3-4 párrafos sobre el tema.
RESTRICCIONES CRÍTICAS:
- Solo usa información de los resultados de búsqueda arriba
- No agregues conocimiento previo
- No menciones que usaste búsqueda web
- Escribe en español claro y accesible
- Cada párrafo debe tener 2-3 oraciones
"""
    
    try:
        message = HumanMessage(content=prompt)
        response = llm.invoke([message])
        state["summary"] = response.content
        print("✓ Resumen generado exitosamente")
        
    except Exception as e:
        print(f"❌ Error generando resumen: {str(e)}")
        state["summary"] = "Error al generar resumen."
    
    state["current_step"] = "generate_question"
    return state

In [ ]:
# NODO 4: Generar pregunta tipo quiz
def generate_question(state: HealthBotState) -> HealthBotState:
    """
    Genera una pregunta tipo quiz basada ÚNICAMENTE en el resumen generado.
    """
    print(f"\n❓ Generando pregunta sobre el tema...")
    
    prompt = f"""Basándote ÚNICAMENTE en el siguiente resumen, genera una pregunta
de opción múltiple (tipo quiz) que pueda ser respondida directamente desde el resumen.

RESUMEN:
{state['summary']}

Genera la pregunta con este formato:
[PREGUNTA]
[RESPUESTA CORRECTA]
[OPCIÓN B (incorrecta pero plausible)]
[OPCIÓN C (incorrecta pero plausible)]
[OPCIÓN D (incorrecta pero plausible)]

La pregunta debe:
- Ser clara y específica
- Ser respondible directamente desde el resumen
- Tener 4 opciones (A, B, C, D)
- Tener una respuesta correcta única
"""
    
    try:
        message = HumanMessage(content=prompt)
        response = llm.invoke([message])
        state["quiz_question"] = response.content
        print("✓ Pregunta generada exitosamente")
        
    except Exception as e:
        print(f"❌ Error generando pregunta: {str(e)}")
        state["quiz_question"] = "Error al generar pregunta."
    
    state["current_step"] = "get_answer"
    return state

In [ ]:
# NODO 5: Obtener respuesta del usuario\ndef get_user_answer(state: HealthBotState) -> HealthBotState:\n    \"\"\"\n    Solicita la respuesta del usuario a la pregunta del quiz.\n    \"\"\"\n    print(\"\\n\" + \"=\"*60)\n    print(\"📖 RESUMEN DEL TEMA\")\n    print(\"=\"*60)\n    print(f\"\\n{state['summary']}\")\n    \n    print(\"\\n\" + \"=\"*60)\n    print(\"❓ PREGUNTA DEL QUIZ\")\n    print(\"=\"*60)\n    print(f\"\\n{state['quiz_question']}\")\n    \n    answer = input(\"\\n✏️ Tu respuesta (A, B, C o D): \").strip().upper()\n    \n    # Validar respuesta\n    while answer not in [\"A\", \"B\", \"C\", \"D\"]:\n        answer = input(\"Por favor, ingresa una opción válida (A, B, C o D): \").strip().upper()\n    \n    state[\"user_answer\"] = answer\n    state[\"current_step\"] = \"grade\"\n    \n    return state"

In [ ]:
# NODO 6: Calificar la respuesta del usuario\ndef grade_answer(state: HealthBotState) -> HealthBotState:\n    \"\"\"\n    Califica la respuesta del usuario usando SOLO el resumen como fuente.\n    Asigna una nota (A, B, C, D, F) y proporciona justificación con citas.\n    \"\"\"\n    print(\"\\n⏳ Evaluando tu respuesta...\")\n    \n    grading_prompt = f\"\"\"Eres un evaluador educativo. Tu tarea es calificar la respuesta del usuario\nbasándote ÚNICAMENTE en el siguiente resumen y pregunta.\n\nRESUMEN (fuente única de verdad):\n{state['summary']}\n\nPREGUNTA DEL QUIZ:\n{state['quiz_question']}\n\nRESPUESTA DEL USUARIO: {state['user_answer']}\n\nTarea:\n1. Determina si la respuesta es correcta comparándola con el resumen\n2. Asigna una calificación: A (excelente), B (bueno), C (satisfactorio), D (insuficiente) o F (fallido)\n3. Proporciona justificación citando fragmentos del resumen\n\nFormato de respuesta:\nCALIFICACIÓN: [A/B/C/D/F]\nJUSTIFICACIÓN: [Tu explicación]\nCITA DEL RESUMEN: \"[Fragmento relevante del resumen]\"\n\"\"\"\n    \n    try:\n        message = HumanMessage(content=grading_prompt)\n        response = llm.invoke([message])\n        \n        # Extraer calificación del response\n        response_text = response.content\n        \n        # Buscar la calificación en el texto\n        grades = {\"A\": \"A\", \"B\": \"B\", \"C\": \"C\", \"D\": \"D\", \"F\": \"F\"}\n        grade_found = None\n        for grade_letter in grades.values():\n            if f\"CALIFICACIÓN: {grade_letter}\" in response_text:\n                grade_found = grade_letter\n                break\n        \n        if not grade_found:\n            # Intenta extraer de otra forma\n            for grade_letter in grades.values():\n                if grade_letter in response_text and \"CALIFICACIÓN\" in response_text:\n                    idx = response_text.find(\"CALIFICACIÓN:\")\n                    if idx != -1:\n                        grade_section = response_text[idx:idx+30]\n                        if grade_letter in grade_section:\n                            grade_found = grade_letter\n                            break\n        \n        state[\"grade\"] = grade_found if grade_found else \"C\"\n        state[\"justification\"] = response_text\n        \n    except Exception as e:\n        print(f\"❌ Error calificando respuesta: {str(e)}\")\n        state[\"grade\"] = \"C\"\n        state[\"justification\"] = f\"Error en evaluación: {str(e)}\"\n    \n    state[\"current_step\"] = \"show_results\"\n    return state"

In [ ]:
# NODO 7: Mostrar resultados\ndef show_results(state: HealthBotState) -> HealthBotState:\n    \"\"\"\n    Muestra los resultados de la evaluación al usuario.\n    \"\"\"\n    print(\"\\n\" + \"=\"*60)\n    print(\"📊 RESULTADOS DE TU EVALUACIÓN\")\n    print(\"=\"*60)\n    \n    # Mapeo de letras a emojis y descripción\n    grade_map = {\n        \"A\": \"🌟 EXCELENTE\",\n        \"B\": \"⭐ MUY BIEN\",\n        \"C\": \"👍 BIEN\",\n        \"D\": \"⚠️ NECESITAS MEJORAR\",\n        \"F\": \"❌ INSUFICIENTE\"\n    }\n    \n    print(f\"\\nTu respuesta: {state['user_answer']}\")\n    print(f\"\\nCALIFICACIÓN: {grade_map.get(state['grade'], state['grade'])}\")\n    print(f\"\\nJUSTIFICACIÓN:\\n{state['justification']}\")\n    \n    state[\"current_step\"] = \"decide_next\"\n    return state"

In [ ]:
# NODO 8: Decidir si continuar o salir\ndef decide_next(state: HealthBotState) -> HealthBotState:\n    \"\"\"\n    Permite al usuario elegir entre:\n    - Aprender otro tema\n    - Salir del sistema\n    \"\"\"\n    print(\"\\n\" + \"=\"*60)\n    print(\"¿QUÉ DESEAS HACER?\")\n    print(\"=\"*60)\n    print(\"\\n1. Aprender un nuevo tema\")\n    print(\"2. Salir del sistema\")\n    \n    choice = input(\"\\n➜ Selecciona una opción (1 o 2): \").strip()\n    \n    while choice not in [\"1\", \"2\"]:\n        choice = input(\"Por favor, selecciona una opción válida (1 o 2): \").strip()\n    \n    state[\"continue_learning\"] = choice == \"1\"\n    \n    if state[\"continue_learning\"]:\n        state[\"current_step\"] = \"input_topic\"\n    else:\n        state[\"current_step\"] = \"end\"\n    \n    return state"

In [ ]:
# CONSTRUCCIÓN DEL GRAFO DE LANGGRAPH\ndef create_healthbot_graph():\n    \"\"\"\n    Crea el grafo de LangGraph con nodos y aristas condicionales.\n    \"\"\"\n    # Inicializar estado\n    workflow = StateGraph(HealthBotState)\n    \n    # Agregar nodos\n    workflow.add_node(\"input_topic\", input_topic)\n    workflow.add_node(\"search_tavily\", search_tavily)\n    workflow.add_node(\"generate_summary\", generate_summary)\n    workflow.add_node(\"generate_question\", generate_question)\n    workflow.add_node(\"get_user_answer\", get_user_answer)\n    workflow.add_node(\"grade_answer\", grade_answer)\n    workflow.add_node(\"show_results\", show_results)\n    workflow.add_node(\"decide_next\", decide_next)\n    \n    # Establecer punto de entrada\n    workflow.set_entry_point(\"input_topic\")\n    \n    # Agregar aristas (transiciones entre nodos)\n    workflow.add_edge(\"input_topic\", \"search_tavily\")\n    workflow.add_edge(\"search_tavily\", \"generate_summary\")\n    workflow.add_edge(\"generate_summary\", \"generate_question\")\n    workflow.add_edge(\"generate_question\", \"get_user_answer\")\n    workflow.add_edge(\"get_user_answer\", \"grade_answer\")\n    workflow.add_edge(\"grade_answer\", \"show_results\")\n    workflow.add_edge(\"show_results\", \"decide_next\")\n    \n    # Arista condicional: ¿continuar o terminar?\n    def should_continue(state):\n        if state[\"continue_learning\"]:\n            return \"input_topic\"  # Reiniciar flujo\n        else:\n            return \"end\"  # Terminar\n    \n    workflow.add_conditional_edges(\n        \"decide_next\",\n        should_continue,\n        {\n            \"input_topic\": \"input_topic\",\n            \"end\": \"__end__\"\n        }\n    )\n    \n    # Compilar el grafo\n    app = workflow.compile()\n    return app\n\n# Crear la aplicación\napp = create_healthbot_graph()\nprint(\"✓ Grafo de LangGraph compilado exitosamente\")"

In [ ]:
# EJECUTAR EL SISTEMA\ndef run_healthbot():\n    \"\"\"\n    Función principal que ejecuta el flujo completo de HealthBot.\n    \"\"\"\n    # Estado inicial\n    initial_state: HealthBotState = {\n        \"topic\": \"\",\n        \"search_results\": [],\n        \"summary\": \"\",\n        \"quiz_question\": \"\",\n        \"user_answer\": \"\",\n        \"grade\": \"\",\n        \"justification\": \"\",\n        \"continue_learning\": False,\n        \"current_step\": \"input_topic\"\n    }\n    \n    try:\n        # Ejecutar el grafo\n        # El grafo ejecutará el flujo completo de forma iterativa\n        # hasta que el usuario elija salir\n        while True:\n            result = app.invoke(initial_state)\n            \n            # Si el usuario no desea continuar, salir del bucle\n            if not result.get(\"continue_learning\", False):\n                break\n            \n            # Actualizar estado para siguiente iteración\n            initial_state = result\n        \n        # Mensaje de despedida\n        print(\"\\n\" + \"=\"*60)\n        print(\"👋 ¡Gracias por usar HealthBot!\")\n        print(\"=\"*60)\n        print(\"\\n¡Que sigas aprendiendo! 📚\")\n        \n    except KeyboardInterrupt:\n        print(\"\\n\\n⚠️ Sesión interrumpida por el usuario\")\n        print(\"\\n👋 ¡Hasta luego!\")\n    except Exception as e:\n        print(f\"\\n❌ Error durante la ejecución: {str(e)}\")\n        import traceback\n        traceback.print_exc()\n\n# Descomenta la siguiente línea para ejecutar el sistema\n# run_healthbot()\n\nprint(\"\\n\" + \"=\"*60)\nprint(\"✅ HealthBot listo para ejecutar\")\nprint(\"=\"*60)\nprint(\"\\nPara iniciar el sistema, ejecuta:\")\nprint(\"  run_healthbot()\")"

In [ ]:
# DEMO: Prueba automática del sistema
print("\n" + "🚀"*30)
print("INICIANDO DEMO DEL SISTEMA HEALTHBOT")
print("🚀"*30)

# Estado inicial con tema de ejemplo
demo_state = {
    "topic": "Inteligencia Artificial",
    "search_results": [],
    "summary": "",
    "quiz_question": "",
    "user_answer": "",
    "grade": "",
    "justification": "",
    "continue_learning": False,
    "current_step": "input_topic"
}

print(f"\n📚 TEMA SELECCIONADO: {demo_state['topic']}\n")

# 1. BÚSQUEDA CON TAVILY
print("="*60)
print("1️⃣ BUSCANDO INFORMACIÓN CON TAVILY...")
print("="*60)
demo_state = search_tavily(demo_state)

# 2. GENERAR RESUMEN
print("\n" + "="*60)
print("2️⃣ GENERANDO RESUMEN EN ESPAÑOL...")
print("="*60)
demo_state = generate_summary(demo_state)
print(f"\n📄 RESUMEN GENERADO:\n{demo_state['summary']}\n")

# 3. GENERAR PREGUNTA
print("="*60)
print("3️⃣ GENERANDO PREGUNTA TIPO QUIZ...")
print("="*60)
demo_state = generate_question(demo_state)
print(f"\n❓ PREGUNTA GENERADA:\n{demo_state['quiz_question']}\n")

# 4. SIMULAR RESPUESTA DEL USUARIO
demo_state["user_answer"] = "A"  # Simular que el usuario responde "A"
print("="*60)
print(f"4️⃣ RESPUESTA DEL USUARIO SIMULADA: {demo_state['user_answer']}")
print("="*60)

# 5. CALIFICAR RESPUESTA
print("\n" + "="*60)
print("5️⃣ CALIFICANDO LA RESPUESTA...")
print("="*60)
demo_state = grade_answer(demo_state)

# 6. MOSTRAR RESULTADOS
print("\n" + "="*60)
print("6️⃣ MOSTRANDO RESULTADOS DE LA EVALUACIÓN...")
print("="*60)
demo_state = show_results(demo_state)

print("\n" + "🎉"*30)
print("✅ DEMO COMPLETADA EXITOSAMENTE")
print("🎉"*30)

print("\n📊 RESUMEN DEL FLUJO EJECUTADO:")
print(f"   ✓ Tema: {demo_state['topic']}")
print(f"   ✓ Fuentes encontradas: {len(demo_state['search_results'])}")
print(f"   ✓ Resumen generado: {len(demo_state['summary'])} caracteres")
print(f"   ✓ Pregunta generada: Sí")
print(f"   ✓ Respuesta del usuario: {demo_state['user_answer']}")
print(f"   ✓ Calificación: {demo_state['grade']}")
print(f"\n🚀 El sistema está 100% FUNCIONAL y listo para usar")
print(f"\nPara usar el sistema interactivamente, ejecuta: run_healthbot()")


## 📖 Guía de Uso

### Requisitos previos
1. **API Keys necesarias**:
   - `OPENAI_API_KEY`: Clave de OpenAI (GPT-4)
   - `TAVILY_API_KEY`: Clave de Tavily Search

2. **Configuración de variables de entorno**:
   Crea un archivo `.env` en el directorio del proyecto:
   ```
   OPENAI_API_KEY=sk-...
   TAVILY_API_KEY=tvly-...
   ```

### Ejecución paso a paso

1. **Ejecuta las celdas en orden** desde la superior a la inferior
2. **Importaciones y configuración**: Se instalarán desde la primera celda
3. **Creación del grafo**: Se compila tras ejecutar todas las celdas de nodos
4. **Ejecución**: Ejecuta esta celda para iniciar:
   ```python
   run_healthbot()
   ```

### Flujo de la aplicación

```
1. 📚 Input Topic
   ↓
2. 🔍 Search Tavily
   ↓
3. 📝 Generate Summary (basado 100% en Tavily)
   ↓
4. ❓ Generate Quiz Question
   ↓
5. ✏️ Get User Answer
   ↓
6. 📊 Grade Answer (usando Solo resumen)
   ↓
7. 📈 Show Results
   ↓
8. 🔄 Decide Next (Continuar o Salir)
```

### Validaciones implementadas

- ✅ Solo Tavily como fuente de información
- ✅ Resumen generado exclusivamente desde búsqueda
- ✅ Pregunta basada únicamente en el resumen
- ✅ Calificación justificada con citas del resumen
- ✅ Manejo de errores en cada etapa
- ✅ Validación de entrada del usuario